# TUGAS MANDIRI PERTEMUAN 5
## Analisis Performa Cabang E-Commerce

**Nama:** MUHAMMAD ZHIO AFFAREL  
**NPM:** 2505060063  
**Kelas:** ROMBEL 2

Notebook ini dikerjakan mengikuti arahan modul: bagian A dan B memakai **DataFrame API**, sedangkan bagian C memakai **Spark SQL murni**.

## Persiapan SparkSession dan Import Library

In [1]:
import os
os.environ["PYSPARK_PYTHON"] = "/home/zhio/miniconda3/envs/bigdata/bin/python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "/home/zhio/miniconda3/envs/bigdata/bin/python"

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, row_number, round as spark_round
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas5-AnalisisCabang") \
    .master("local[*]") \
    .config("spark.pyspark.python", "/home/zhio/miniconda3/envs/bigdata/bin/python") \
    .config("spark.pyspark.driver.python", "/home/zhio/miniconda3/envs/bigdata/bin/python") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("SparkSession siap. Versi Spark:", spark.version)

26/09/23 06:40:52 WARN Utils: Your hostname, vmi3356457 resolves to a loopback address: 127.0.1.1; using 13.140.149.59 instead (on interface eth0)
26/09/23 06:40:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/23 06:40:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


## 1. Membaca Data Transaksi dari HDFS

Data transaksi dibaca langsung dari lokasi HDFS yang ditentukan modul. Setelah itu, kolom `pendapatan` dihitung dari `unit_terjual × harga_satuan`.

In [2]:
path_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv"

df_transaksi = spark.read.csv(path_hdfs, header=True, inferSchema=True)
df_transaksi = df_transaksi.withColumn(
    "pendapatan", col("unit_terjual") * col("harga_satuan")
)

print("Jumlah transaksi:", df_transaksi.count())
df_transaksi.printSchema()
df_transaksi.show(5)

Jumlah transaksi: 500
root
 |-- order_id: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- pendapatan: integer (nullable = true)



+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows



## 2. Membuat DataFrame Target Cabang

In [3]:
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

import pandas as pd
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))
df_target.show()

+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



## A. Join dan Perbandingan Target (DataFrame API)

Saya meringkas total pendapatan per kota, melakukan `join` dengan tabel target, lalu menghitung persentase pencapaian terhadap target bulanan.

In [4]:
ringkasan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

hasil_a = ringkasan_kota.join(df_target, on="kota", how="inner") \
    .withColumn(
        "pencapaian_persen",
        spark_round(col("total_pendapatan") / col("target_bulanan") * 100, 2)
    ) \
    .select("kota", "pic_cabang", "total_pendapatan",
            "target_bulanan", "pencapaian_persen") \
    .orderBy(col("pencapaian_persen").desc())

hasil_a.show(truncate=False)

+----------+----------+----------------+--------------+-----------------+
|kota      |pic_cabang|total_pendapatan|target_bulanan|pencapaian_persen|
+----------+----------+----------------+--------------+-----------------+
|Purworejo |Fitri     |45650000        |30000000      |152.17           |
|Solo      |Bayu      |33475000        |40000000      |83.69            |
|Yogyakarta|Joko      |47275000        |60000000      |78.79            |
|Magelang  |Rani      |31650000        |45000000      |70.33            |
|Semarang  |Sari      |38175000        |55000000      |69.41            |
+----------+----------+----------------+--------------+-----------------+



## B. Window Function — Kategori Terlaris per Kota (DataFrame API)

Pendapatan terlebih dahulu dijumlahkan berdasarkan kota dan kategori. Selanjutnya, `row_number()` digunakan untuk memberikan urutan dalam setiap kota, dan hanya peringkat pertama yang ditampilkan.

In [5]:
pendapatan_kategori = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan_kategori")
)

window_kota = Window.partitionBy("kota").orderBy(
    col("total_pendapatan_kategori").desc(), col("kategori").asc()
)

hasil_b = pendapatan_kategori.withColumn(
    "peringkat", row_number().over(window_kota)
).filter(col("peringkat") == 1).orderBy("kota")

hasil_b.show(truncate=False)

+----------+----------------------+-------------------------+---------+
|kota      |kategori              |total_pendapatan_kategori|peringkat|
+----------+----------------------+-------------------------+---------+
|Magelang  |Kesehatan & Kecantikan|7275000                  |1        |
|Purworejo |Kesehatan & Kecantikan|10075000                 |1        |
|Semarang  |Rumah Tangga          |11125000                 |1        |
|Solo      |Kesehatan & Kecantikan|8425000                  |1        |
|Yogyakarta|Fashion               |13325000                 |1        |
+----------+----------------------+-------------------------+---------+



## C. Spark SQL

Kedua DataFrame didaftarkan sebagai temporary view. Perhitungan berikut dilakukan dengan satu kueri SQL, bukan DataFrame API.

In [6]:
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target_cabang")

hasil_c = spark.sql("""
    SELECT
        t.kota,
        d.pic_cabang,
        COUNT(*) AS jumlah_transaksi
    FROM transaksi t
    JOIN target_cabang d ON t.kota = d.kota
    GROUP BY t.kota, d.pic_cabang
    ORDER BY jumlah_transaksi DESC, t.kota ASC
""")

hasil_c.show(truncate=False)

+----------+----------+----------------+
|kota      |pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
|Purworejo |Fitri     |116             |
|Yogyakarta|Joko      |110             |
|Solo      |Bayu      |95              |
|Semarang  |Sari      |93              |
|Magelang  |Rani      |86              |
+----------+----------+----------------+



## D. Kesimpulan

Berdasarkan hasil analisis bagian A, cabang **Purworejo** yang dikelola oleh **Fitri** merupakan cabang dengan kinerja paling baik. Cabang ini memperoleh total pendapatan sebesar **Rp45.650.000**, melampaui target bulanan **Rp30.000.000**, sehingga pencapaiannya mencapai **152,17%**. Berdasarkan bagian B, kategori dengan pendapatan tertinggi di Purworejo adalah **Kesehatan & Kecantikan**, dengan total **Rp10.075.000**. Cabang yang paling perlu mendapat perhatian manajemen adalah **Semarang** yang dikelola oleh **Sari**. Pendapatannya sebesar **Rp38.175.000** dari target **Rp55.000.000**, sehingga pencapaiannya hanya **69,41%**, paling rendah di antara seluruh cabang. Kategori teratas di Semarang adalah **Rumah Tangga** dengan pendapatan **Rp11.125.000**. Meskipun Semarang mempunyai 93 transaksi, pencapaian targetnya tetap rendah. Manajemen perlu mengevaluasi nilai transaksi rata-rata, strategi promosi, komposisi kategori, dan kesesuaian target cabang Semarang. Strategi kategori unggulan dapat dipertahankan, sedangkan kategori lain perlu ditingkatkan melalui promosi yang lebih terarah.

## Pemeriksaan Akhir

In [7]:
print("Jumlah baris hasil A:", hasil_a.count())
print("Jumlah kota top-1 hasil B:", hasil_b.count())
print("Jumlah baris hasil C:", hasil_c.count())
assert hasil_a.count() == 5
assert hasil_b.count() == 5
assert hasil_c.count() == 5
print("VALIDASI BERHASIL: bagian A, B, dan C masing-masing menghasilkan 5 kota.")

Jumlah baris hasil A: 5


Jumlah kota top-1 hasil B: 5


Jumlah baris hasil C: 5


VALIDASI BERHASIL: bagian A, B, dan C masing-masing menghasilkan 5 kota.


In [8]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
